# 选修E3 · Day 3 上机：LLM 评估与部署

**版本**：v5.0 学习材料包
**配套**：notes.md（讲义）｜ data/README.md（真实库/数据）｜ solution.ipynb（参考答案，做完再看）

**核心命题**：营销 LLM 上线后，用 deepeval 评估质量、用 langsmith 追踪调用、用 tiktoken 监控成本、用 vLLM/投机解码/MoE 优化推理。

**真实库**：deepeval（评估指标）+ langsmith（追踪）+ tiktoken（token 成本）


## 0. 环境准备

安装并导入真实库：
- **deepeval**：LLM 评估框架，自定义 BaseMetric + LLMTestCase
- **langsmith**：LLM 应用追踪，@traceable 装饰器
- **tiktoken**：token 计数，成本监控基础

In [ ]:
# !pip install deepeval langsmith tiktoken -q

import tiktoken
from dataclasses import dataclass
from typing import List
from deepeval.metrics import BaseMetric
from deepeval.test_case import LLMTestCase
from langsmith import traceable
import time

print("=== 环境就绪 ===")
print(f"tiktoken: {tiktoken.__version__}")
print("deepeval: 可用 (自定义 BaseMetric + LLMTestCase)")
print("langsmith: 可用 (@traceable 本地模式)")


## 1. 场景背景与营销映射

**场景**：智能手表品牌上线了一个营销文案 LLM（自动生成小红书/朋友圈/电商详情页文案）。现在需要：
1. 构建评估集，评估文案质量（准确性/相关性/无害性/忠实性）
2. 追踪部署后每次调用的延迟和 token
3. 监控日均万次请求的推理成本
4. 优化推理架构（vLLM/投机解码/MoE 概念）

**评估集**：5 条真实营销文案样本，覆盖优质/错误/违规/幻觉/跑题五种失败模式。

In [ ]:
# 营销文案评估集（基于真实智能手表产品文档结构构建，见 data/README.md）
@dataclass
class EvalExample:
    input: str              # 输入（产品信息/用户需求）
    actual_output: str      # 待评估的 LLM 生成文案
    expected_output: str    # 参考答案（优质文案）
    criteria: List[str]     # 评估标准
    category: str           # 任务类别

marketing_eval_set = [
    EvalExample(
        input="为智能手表Pro写小红书种草文案，卖点：7天续航、100+运动模式、心率血氧",
        actual_output="姐妹们！这款智能手表Pro真的绝了！续航7天不充电，100+运动模式全覆盖，心率血氧24小时守护。运动女孩必备！快冲！",
        expected_output="告别一天一充的焦虑！智能手表Pro续航7天，100+运动模式，心率血氧实时监测，让你的健康尽在掌握。",
        criteria=["含产品卖点", "符合小红书风格", "有CTA"],
        category="小红书种草",
    ),
    EvalExample(
        input="为智能手表Pro写电商详情页文案",
        actual_output="这款手表续航7天，支持100+运动模式，价格只要99元，还送AirPods。",
        expected_output="智能手表Pro，7天超长续航，100+运动模式，心率血氧监测，5ATM防水。官方售价1299元。",
        criteria=["参数准确", "无虚假宣传", "含价格"],
        category="电商详情页",
    ),
    EvalExample(
        input="为智能手表Pro写朋友圈广告",
        actual_output="这款手表只适合北方人用，南方人别买。",
        expected_output="智能手表Pro，守护全家健康。7天续航，100+运动模式，心率血氧监测。送父母首选。",
        criteria=["无歧视", "无违规", "通用性"],
        category="朋友圈广告",
    ),
    EvalExample(
        input="基于产品知识库写文案，知识库：续航7天、100+运动、5ATM防水",
        actual_output="这款手表续航30天，支持200+运动模式，10ATM防水，还能测血压。",
        expected_output="智能手表Pro续航7天，100+运动模式，5ATM防水，满足日常运动健康需求。",
        criteria=["忠于知识库", "无幻觉", "参数准确"],
        category="RAG生成",
    ),
    EvalExample(
        input="为智能手表Pro写种草文案",
        actual_output="好。",
        expected_output="智能手表Pro，7天续航+100+运动模式+心率血氧，让你的健康尽在掌握。",
        criteria=["含产品卖点", "有CTA", "符合风格"],
        category="小红书种草",
    ),
]

print(f"评测集样本数: {len(marketing_eval_set)}")
for i, ex in enumerate(marketing_eval_set):
    print(f"  [{i+1}] {ex.category}: {ex.actual_output[:35]}...")


## TODO 1：把营销评测集转换为 deepeval LLMTestCase

deepeval 的评估对象是 `LLMTestCase`，需要把上面的 `EvalExample` 转换：
- `input` <- EvalExample.input
- `actual_output` <- EvalExample.actual_output（待评估文案）
- `expected_output` <- EvalExample.expected_output（参考答案）
- `context` <- 提取知识库信息（用于忠实性评估）

**任务**：写 `to_test_cases(eval_set)` 函数，返回 `List[LLMTestCase]`。

In [ ]:
# 把 EvalExample 转换为 deepeval LLMTestCase
def to_test_cases(eval_set):
    """把营销评测集转换为 deepeval LLMTestCase 列表"""
    test_cases = []
    for ex in eval_set:
        tc = LLMTestCase(
            input=ex.input,
            actual_output=ex.actual_output,
            expected_output=ex.expected_output,
            context=[ex.expected_output],  # 用参考答案作为上下文（忠实性评估）
        )
        test_cases.append(tc)
    return test_cases

test_cases = to_test_cases(marketing_eval_set)
print(f"=== LLMTestCase 转换完成 ===")
print(f"测试用例数: {len(test_cases)}")
for i, tc in enumerate(test_cases):
    print(f"  [{i+1}] input={tc.input[:30]}... | actual={tc.actual_output[:30]}...")


## TODO 2：用 deepeval 自定义 MarketingQualityMetric

继承 `BaseMetric`，实现四维度规则评分（无 API，规则近似 LLM-as-a-Judge）：

| 维度 | 规则 | 评分逻辑 |
|------|------|---------|
| **准确性** | actual 与 expected 的关键词重合度 | 重合度高=1.0，中=0.5，低=0.0 |
| **相关性** | actual 是否包含 input 的核心词 | 包含=1.0，部分=0.5，无=0.0 |
| **无害性** | actual 是否含违禁词（歧视/违规） | 无违禁=1.0，有=0.0 |
| **忠实性** | actual 是否忠于 context（无幻觉） | 参数与 context 一致=1.0，编造=0.0 |

**任务**：实现 `measure(test_case)` 和 `is_successful()`。

In [ ]:
# 用 deepeval 自定义 MarketingQualityMetric（四维度规则评分）
import re

class MarketingQualityMetric(BaseMetric):
    """营销文案质量评估指标（四维度规则评分，无 API，LLM-as-a-Judge fallback）"""

    def __init__(self, threshold: float = 0.6):
        self.threshold = threshold
        self.score = 0.0
        self.score_breakdown = {}
        self.reason = ""
        self.success = False

    def measure(self, test_case, *args, **kwargs):
        actual = test_case.actual_output or ""
        expected = test_case.expected_output or ""
        context = " ".join(test_case.context) if test_case.context else ""
        inp = test_case.input or ""

        # 维度1：准确性 —— actual 与 expected 关键词重合度
        actual_words = set(re.findall(r'[\w\u4e00-\u9fff]+', actual.lower()))
        expected_words = set(re.findall(r'[\w\u4e00-\u9fff]+', expected.lower()))
        overlap = len(actual_words & expected_words) / max(len(expected_words), 1)
        accuracy = 1.0 if overlap >= 0.4 else (0.5 if overlap >= 0.15 else 0.0)

        # 维度2：相关性 —— actual 是否包含 input 核心词
        input_words = set(re.findall(r'[\u4e00-\u9fff]{2,}', inp))
        hits = sum(1 for w in input_words if w in actual)
        relevance = 1.0 if hits >= 2 else (0.5 if hits >= 1 else 0.0)

        # 维度3：无害性 -- 是否含违禁词（地域/种族歧视）
        forbidden = ["北方人", "南方人", "歧视", "黑人", "白人"]
        has_forbidden = any(w in actual for w in forbidden)
        harmlessness = 0.0 if has_forbidden else 1.0

        # 维度4：忠实性 -- actual 规格参数是否忠于 context（防幻觉）
        # 检测 actual 中的数字，若不在 input/expected/context 且附近有规格关键词，则判幻觉
        all_context = context + " " + inp + " " + expected
        all_context_nums = set(re.findall(r'\d+', all_context))
        spec_keywords = ['续航', '运动', '防水', 'ATM', '天', '模式', '价格', '元', '血氧', '心率']
        actual_num_list = re.findall(r'\d+', actual)
        hallucinated = []
        for num in actual_num_list:
            if num not in all_context_nums:
                idx = actual.find(num)
                window = actual[max(0, idx - 10):idx + len(num) + 10]
                if any(kw in window for kw in spec_keywords):
                    hallucinated.append(num)
        faithfulness = 0.0 if hallucinated else 1.0

        self.score_breakdown = {
            "accuracy": accuracy,
            "relevance": relevance,
            "harmlessness": harmlessness,
            "faithfulness": faithfulness,
        }
        self.score = sum(self.score_breakdown.values()) / 4
        self.reason = f"accuracy={accuracy:.1f}, relevance={relevance:.1f}, harmlessness={harmlessness:.1f}, faithfulness={faithfulness:.1f}"
        self.success = self.score >= self.threshold
        return self.score

    def is_successful(self):
        return self.success

    @property
    def __name__(self):
        return "MarketingQualityMetric"

# 测试单条
metric = MarketingQualityMetric(threshold=0.6)
metric.measure(test_cases[0])
print(f"=== 单条评估测试（样本1）===")
print(f"总分: {metric.score:.2f} (阈值 {metric.threshold})")
print(f"分维度: {metric.score_breakdown}")
print(f"通过: {metric.is_successful()}")
print(f"理由: {metric.reason}")


## TODO 3：批量评估营销文案集，输出评分矩阵

用自定义 metric 对所有 LLMTestCase 批量评估，输出每条文案的四维度评分和总分。

**任务**：写 `evaluate_all(test_cases, metric)` 函数，返回评分矩阵（list of dict）。

> 注：deepeval 也提供 `from deepeval import evaluate` 的批量 API，但需要配置评估模型。本任务用手动循环 `metric.measure(tc)` 实现规则评估（无 API），效果等价。

In [ ]:
# 批量评估营销文案集
def evaluate_all(test_cases, metric):
    """批量评估，返回评分矩阵"""
    results = []
    for i, tc in enumerate(test_cases):
        metric.measure(tc)
        row = {
            "id": i + 1,
            "category": marketing_eval_set[i].category,
            "actual": tc.actual_output[:40] + "...",
            **metric.score_breakdown,
            "total": metric.score,
            "passed": metric.is_successful(),
        }
        results.append(row)
    return results

results = evaluate_all(test_cases, MarketingQualityMetric(threshold=0.6))

print("=== 营销文案评估矩阵 ===")
print(f"{'ID':>3} {'类别':12s} {'准确':>5} {'相关':>5} {'无害':>5} {'忠实':>5} {'总分':>5} {'通过':>5}")
print("-" * 60)
for r in results:
    print(f"{r['id']:>3} {r['category']:12s} {r['accuracy']:>5.1f} {r['relevance']:>5.1f} {r['harmlessness']:>5.1f} {r['faithfulness']:>5.1f} {r['total']:>5.2f} {'✓' if r['passed'] else '✗':>5}")

pass_rate = sum(r['passed'] for r in results) / len(results)
print(f"\n通过率: {pass_rate*100:.0f}% ({sum(r['passed'] for r in results)}/{len(results)})")
avg_scores = {k: sum(r[k] for r in results)/len(results) for k in ['accuracy','relevance','harmlessness','faithfulness']}
print(f"平均分维度: {avg_scores}")


## TODO 4：用 langsmith @traceable 追踪部署后营销 LLM 调用

模拟部署后的营销文案生成 LLM（mock，无真实 API），用 `@traceable` 装饰器追踪调用链：
- 记录输入（产品信息）
- 记录输出（生成文案）
- 记录延迟（模拟推理时间）

**任务**：用 `@traceable` 装饰 `marketing_llm_generate` 函数，运行 3 次调用并打印 trace 信息。

> 注：无 `LANGSMITH_API_KEY` 时 `@traceable` 仍可运行（本地模式，trace 存内存）。

In [ ]:
# 用 langsmith @traceable 追踪部署后营销 LLM 调用（mock LLM）
import random

@traceable(name="marketing_llm_generate")
def marketing_llm_generate(product_brief: str, style: str = "小红书") -> dict:
    """模拟部署后的营销文案生成 LLM（mock，无真实 API 调用）"""
    start = time.perf_counter()
    # 模拟 LLM 推理延迟（100-300ms）
    time.sleep(random.uniform(0.1, 0.3))
    # mock 生成
    templates = {
        "小红书": f"姐妹们！{product_brief}真的绝了！快冲！",
        "朋友圈": f"推荐：{product_brief}，守护全家健康。",
        "电商": f"{product_brief}，7天续航，100+运动模式，5ATM防水。",
    }
    output = templates.get(style, templates["小红书"])
    latency = (time.perf_counter() - start) * 1000
    return {
        "input": product_brief,
        "style": style,
        "output": output,
        "latency_ms": round(latency, 1),
        "tokens_in": len(product_brief),
        "tokens_out": len(output),
    }

# 运行 3 次调用
briefs = [
    ("智能手表Pro 7天续航 100+运动", "小红书"),
    ("智能手表Pro 心率血氧", "朋友圈"),
    ("智能手表Pro 5ATM防水", "电商"),
]

print("=== LangSmith 追踪营销 LLM 调用 ===")
traces = []
for brief, style in briefs:
    result = marketing_llm_generate(brief, style)
    traces.append(result)
    print(f"[{style:5s}] latency={result['latency_ms']:.1f}ms | out={result['output'][:30]}...")

avg_latency = sum(t['latency_ms'] for t in traces) / len(traces)
total_tokens_out = sum(t['tokens_out'] for t in traces)
print(f"\n调用次数: {len(traces)}")
print(f"平均延迟: {avg_latency:.1f}ms")
print(f"总输出 tokens (近似): {total_tokens_out}")
print("（@traceable 已记录 trace，无 API key 时存本地内存）")


## TODO 5：用 tiktoken 监控日均万次营销文案生成的 token 成本

营销 LLM 部署后日均 10000 次调用，需要监控推理成本：
- 用 tiktoken 精确统计 input/output token（gpt-4o 用 `o200k_base`，DeepSeek V3 用 `cl100k_base`）
- 结合模型定价计算日均/月均成本
- 对比 gpt-4o vs DeepSeek V3（MoE 架构，成本仅 1/10）

**任务**：实现 `estimate_cost(text_in, text_out, daily_calls, model)` 函数。

In [ ]:
# 用 tiktoken 监控部署后 token 成本
def estimate_cost(text_in: str, text_out: str, daily_calls: int, model: str) -> dict:
    """估算日均/月均推理成本"""
    encoders = {
        "gpt-4o":      ("o200k_base",  2.50, 10.00),
        "DeepSeek V3": ("cl100k_base", 0.27, 1.10),
        "gpt-4o-mini": ("o200k_base",  0.15, 0.60),
    }
    enc_name, p_in, p_out = encoders[model]
    enc = tiktoken.get_encoding(enc_name)
    tokens_in = len(enc.encode(text_in))
    tokens_out = len(enc.encode(text_out))

    daily_cost = (tokens_in * daily_calls / 1_000_000 * p_in +
                  tokens_out * daily_calls / 1_000_000 * p_out)
    monthly_cost = daily_cost * 30

    return {
        "model": model,
        "tokens_in": tokens_in,
        "tokens_out": tokens_out,
        "daily_calls": daily_calls,
        "daily_cost": round(daily_cost, 2),
        "monthly_cost": round(monthly_cost, 2),
    }

# 营销文案样本
sample_input = "为智能手表Pro写小红书种草文案，卖点：7天续航、100+运动模式、心率血氧监测、5ATM防水。"
sample_output = "姐妹们！这款智能手表Pro真的绝了！续航7天不充电，100+运动模式全覆盖，心率血氧24小时守护，5ATM防水游泳无忧。运动女孩必备！快冲！"

daily_calls = 10000  # 日均万次

print("=== 部署后 token 成本监控（日均 10000 次调用）===")
print(f"输入样本: {sample_input[:30]}... ({len(sample_input)} chars)")
print(f"输出样本: {sample_output[:30]}... ({len(sample_output)} chars)")
print(f"日均调用: {daily_calls} 次\n")

models = ["gpt-4o", "gpt-4o-mini", "DeepSeek V3"]
results = {m: estimate_cost(sample_input, sample_output, daily_calls, m) for m in models}

print(f"{'模型':14s} {'in_tokens':>10} {'out_tokens':>11} {'日成本':>10} {'月成本':>10}")
print("-" * 60)
for m in models:
    r = results[m]
    print(f"{m:14s} {r['tokens_in']:>10} {r['tokens_out']:>11} ${r['daily_cost']:>8.2f} ${r['monthly_cost']:>8.2f}")

gpt4o_monthly = results["gpt-4o"]["monthly_cost"]
ds_monthly = results["DeepSeek V3"]["monthly_cost"]
print(f"\nDeepSeek V3 月成本仅为 gpt-4o 的 {ds_monthly/gpt4o_monthly*100:.1f}%")
print(f"节省: ${gpt4o_monthly - ds_monthly:.2f}/月（{(1-ds_monthly/gpt4o_monthly)*100:.1f}%）")
print("\n（DeepSeek V3 用 MoE 架构 671B 总参数/37B 激活，推理成本大幅降低）")


## TODO 6：用 LLM-as-a-Judge 规则近似实现自动评分

LLM-as-a-Judge 通常用强 LLM（如 GPT-4）评判弱 LLM 输出。无 API 时，用规则近似实现：
- 关键词匹配（卖点覆盖）
- 长度检测（过短=跑题）
- CTA 检测（有无行动号召）
- 违禁词检测（无害性）

**任务**：实现 `LLMJudge` 类，对每条文案自动评分并输出诊断报告。

In [ ]:
# LLM-as-a-Judge 规则近似实现（无 API）
class LLMJudge:
    """LLM-as-a-Judge 规则近似（无 API，关键词/长度/CTA/违禁词检测）"""

    def __init__(self):
        self.forbidden_words = ["北方人", "南方人", "歧视", "黑人", "白人"]
        self.cta_signals = ["快冲", "必买", "推荐", "首选", "赶紧", "点击", "下单", "入手"]

    def judge(self, actual: str, expected: str, input_text: str) -> dict:
        # 1. 卖点覆盖（actual vs expected 关键词）
        actual_words = set(re.findall(r'[\u4e00-\u9fff]{2,}', actual))
        expected_words = set(re.findall(r'[\u4e00-\u9fff]{2,}', expected))
        coverage = len(actual_words & expected_words) / max(len(expected_words), 1)

        # 2. 长度检测（过短=跑题）
        length = len(actual)
        length_score = 1.0 if length >= 20 else (0.5 if length >= 10 else 0.0)

        # 3. CTA 检测
        has_cta = any(cta in actual for cta in self.cta_signals)

        # 4. 违禁词检测
        has_forbidden = any(w in actual for w in self.forbidden_words)

        # 5. 幻觉检测（actual 数字 vs expected 数字）
        all_text = expected + ' ' + input_text
        all_nums = set(re.findall(r'\d+', all_text))
        spec_keywords = ['续航', '运动', '防水', 'ATM', '天', '模式', '价格', '元', '血氧', '心率']
        actual_num_set = set(re.findall(r'\d+', actual))
        hallucinated_nums = []
        for num in actual_num_set:
            if num not in all_nums:
                idx = actual.find(num)
                window = actual[max(0, idx - 10):idx + len(num) + 10]
                if any(kw in window for kw in spec_keywords):
                    hallucinated_nums.append(num)

        # 综合评分
        scores = {
            "卖点覆盖": min(coverage * 2, 1.0),
            "长度适中": length_score,
            "有CTA": 1.0 if has_cta else 0.0,
            "无违禁": 0.0 if has_forbidden else 1.0,
            "无幻觉": 0.0 if hallucinated_nums else 1.0,
        }
        total = sum(scores.values()) / len(scores)

        return {
            "scores": scores,
            "total": total,
            "diagnosis": {
                "长度": length,
                "卖点覆盖数": len(actual_words & expected_words),
                "有CTA": has_cta,
                "违禁词": [w for w in self.forbidden_words if w in actual],
                "幻觉数字": list(hallucinated_nums),
            },
            "verdict": "通过" if total >= 0.6 and not has_forbidden else "不通过",
        }

# 对所有评估集样本自动评分
judge = LLMJudge()
print("=== LLM-as-a-Judge 自动评分报告 ===")
print(f"{'ID':>3} {'类别':12s} {'覆盖':>5} {'长度':>5} {'CTA':>5} {'无害':>5} {'忠实':>5} {'总分':>5} {'判定':>6}")
print("-" * 70)
for i, ex in enumerate(marketing_eval_set):
    r = judge.judge(ex.actual_output, ex.expected_output, ex.input)
    s = r['scores']
    print(f"{i+1:>3} {ex.category:12s} {s['卖点覆盖']:>5.1f} {s['长度适中']:>5.1f} {s['有CTA']:>5.1f} {s['无违禁']:>5.1f} {s['无幻觉']:>5.1f} {r['total']:>5.2f} {r['verdict']:>6}")

print("\n=== 诊断详情（样本3 违规文案）===")
r3 = judge.judge(marketing_eval_set[2].actual_output, marketing_eval_set[2].expected_output, marketing_eval_set[2].input)
print(f"文案: {marketing_eval_set[2].actual_output}")
print(f"诊断: {r3['diagnosis']}")
print(f"判定: {r3['verdict']}")

print("\n=== 诊断详情（样本4 幻觉文案）===")
r4 = judge.judge(marketing_eval_set[3].actual_output, marketing_eval_set[3].expected_output, marketing_eval_set[3].input)
print(f"文案: {marketing_eval_set[3].actual_output}")
print(f"诊断: {r4['diagnosis']}")
print(f"判定: {r4['verdict']}")


## 总结

### 你完成了什么

1. **构建营销领域评测集**：5 条真实文案样本，覆盖优质/错误/违规/幻觉/跑题五种失败模式
2. **自定义 deepeval BaseMetric**：四维度规则评分（准确性/相关性/无害性/忠实性），无 API 的 LLM-as-a-Judge fallback
3. **批量评估营销文案**：输出评分矩阵，识别质量瓶颈
4. **LangSmith 追踪部署后 LLM**：@traceable 记录调用链、延迟、token
5. **tiktoken 监控推理成本**：日均万次请求的 gpt-4o vs DeepSeek V3 成本对比
6. **LLM-as-a-Judge 规则近似**：自动评分 + 诊断报告（卖点/长度/CTA/违禁/幻觉）

### 关键收获

- **评估三层框架**：通用基准（MMLU/HumanEval/AgentBench）→ 任务评测集 → 系统效果 A/B
- **deepeval 是 LLM 评估的 pytest**：自定义 BaseMetric 可嵌入 CI/CD
- **LangSmith 是 LLM 应用的 APM**：没有追踪的 LLM 应用等于黑箱
- **推理成本是部署核心瓶颈**：DeepSeek V3（MoE）用 1/10 成本逼近 gpt-4o 质量

### 2026 前沿关键词

`deepeval` · `LLM-as-a-Judge` · `LangSmith` · `vLLM` · `投机解码` · `MoE` · `AgentBench` · `RAGAS`

### 下一步

- 用 deepeval 评估自己构建的营销 RAG 系统（Day 2 的 RAGAS 指标可复用）
- 设计 A/B 测试验证 LLM 优化的业务价值
- 阅读 vLLM / 投机解码论文，理解推理优化架构